In [ ]:
import pandas as pd
import numpy as np

expert = pd.read_excel('ExpertReviewsClean43LIWC.xlsx')
expert.head()

# 1. Checking Data Types

In [ ]:
expert.info()
expert.describe()

# Shape 

238,973 rows x 98 columns

# Duplicate Rows

In [ ]:
print("Duplicate Rows:", expert.duplicated().sum())

In [ ]:
before = len(expert)
expert = expert.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(expert)} rows; {len(expert)} remain")

In [ ]:
print("Duplicate Rows:", expert.duplicated().sum())

# NaN Values

In [ ]:
na = expert.isna().sum()
na[na > 0] #instead of 98 lines we get a short list of only the columns with missing values

In [ ]:
expert[expert.isna().any(axis=1)] #which rows have NaN values? 

In [ ]:
expert = expert.dropna(subset=['idvscore', 'reviewer', 'dateP', 'Rev'])

In [ ]:
expert = expert.reset_index(drop=True)
print("Real NaN values left:", expert.isna().sum().sum())
print("Rows:", len(expert))

# Find Hidden Missing Values e.g "None"
### Pandas can't tell that the text "None" means missing, so it counts "None" as a real reviewer name or date

In [ ]:
text_cols = expert.select_dtypes(include=['object', 'str']).columns
placeholders = ['', 'nan', 'none', 'null', 'n/a', 'na', '-', '?']

for col in text_cols:
    s = expert[col].astype(str).str.strip().str.strip("'\"").str.strip().str.lower()
    found = s[s.isin(placeholders)].value_counts()
    if len(found):
        print(col, found.to_dict())


In [ ]:
for col in ['reviewer', 'dateP', 'Rev']:
    s = expert[col].astype(str).str.strip().str.strip("'\"").str.strip().str.lower()
    expert.loc[s.isin(['none', '']), col] = np.nan

In [ ]:
expert[['reviewer', 'dateP', 'Rev']].isna().sum()


In [ ]:
print("Total rows:", len(expert))
(expert[['reviewer', 'dateP', 'Rev']].isna().mean() * 100).round(1)
#In words, "For each of the three columns, what percentage of cells are blank?"

# Data Types

In [ ]:
expert['idvscore'] = expert['idvscore'].astype(int)


In [ ]:
expert.info()
expert.dtypes

### Dates 

In [ ]:
expert['dateP'].dropna().sample(10, random_state=1)
#see what the dates look like, while skipping the blanks.

### It's the same thing we saw with 'None' in Rev: the quote marks are part of the text. We need to remove them first, or pandas won't recognise the dates.

In [ ]:
expert['dateP'] = pd.to_datetime(
    expert['dateP'].str.strip().str.strip("'"),
    format='%b %d, %Y'
)

In [ ]:
print(expert['dateP'].isna().sum())
expert['dateP'].dtype

In [ ]:
expert.head()

### Everything is looking good , but reviewer and Rev columns still have quotes and stuff, so we have to clean that up.

In [ ]:
for col in ['reviewer', 'Rev']:
    expert[col] = (
        expert[col]
        .str.strip()
        .str.replace(r"(?s)^(['\"])(.*)\1$", r"\2", regex=True)
    )

In [ ]:
for col in ['reviewer', 'Rev']:
    print(col, expert[col].str.match(r"^['\"]").sum())

In [ ]:
expert.head(3)

# Surrogate Key
### Add id_expert_review as the primary key, after all rows have been removed so the IDs are 1..n with no gaps.

In [ ]:
if 'id_expert_review' not in expert.columns: #so re-running this cell doesn't crash
    expert.insert(0, 'id_expert_review', range(1, len(expert) + 1))

assert expert['id_expert_review'].is_unique
expert[['id_expert_review', 'url', 'reviewer']].head()

In [ ]:
with pd.ExcelWriter('expert_cleaned.xlsx', datetime_format='yyyy-mm-dd') as writer:
    expert.to_excel(writer, sheet_name='expert', index=False)

print("Saved:", expert.shape)

In [ ]:
check = pd.read_excel('expert_cleaned.xlsx')
print("Shape:", check.shape)
print("id_expert_review unique:", check['id_expert_review'].is_unique)
print(check.dtypes[['id_expert_review', 'idvscore', 'reviewer', 'dateP', 'Rev']])
check[['reviewer', 'dateP', 'Rev']].isna().sum()